# BITS WILP M.Tech (AIML/DSE) - Machine Learning Assignment 2
## Mobile Price Classification - Model Training & Hyperparameter Tuning

This notebook covers the machine learning pipeline for the Mobile Price Classification dataset. The objective is to predict the price range of a mobile phone (0: Low Cost, 1: Medium Cost, 2: High Cost, 3: Very High Cost) using various features. We will train, fine-tune, and evaluate 5 different classification algorithms on the same dataset:

1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **K-Nearest Neighbors (KNN) Classifier**
4. **Naive Bayes Classifier (Gaussian)**
5. **Random Forest Classifier (Ensemble)**

For each model, we compute the following 6 evaluation metrics:
* Accuracy
* AUC Score (One-vs-Rest for multi-class)
* Precision (Macro average)
* Recall (Macro average)
* F1 Score (Macro average)
* Matthews Correlation Coefficient (MCC)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

%matplotlib inline
sns.set_theme(style="whitegrid")

### 1. Load Dataset & Explore

In [ ]:
# Load dataset
df = pd.read_csv('../mobile_data.csv')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Display class distribution
print("Class distribution in price_range:")
print(df['price_range'].value_counts())
df.info()

### 2. Data Preparation & Scaling

In [ ]:
# Separate features and target
X = df.drop('price_range', axis=1)
y = df['price_range']

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler for Streamlit app deployment
joblib.dump(scaler, 'scaler.joblib')
print("Scaler saved successfully.")

### 3. Model Training & Hyperparameter Tuning

In [ ]:
# Define models and parameter grids
models_config = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'grid': {
            'C': [0.01, 0.1, 1.0, 10.0, 100.0],
            'solver': ['lbfgs', 'saga']
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'grid': {
            'max_depth': [3, 5, 8, 12, None],
            'min_samples_split': [2, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    },
    'kNN': {
        'model': KNeighborsClassifier(),
        'grid': {
            'n_neighbors': [3, 5, 7, 11, 15],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    },
    'Naive Bayes': {
        'model': GaussianNB(),
        'grid': {
            'var_smoothing': np.logspace(0, -9, num=100)
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'grid': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 15, None],
            'min_samples_split': [2, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    }
}

results = []
confusion_matrices = {}

for name, config in models_config.items():
    print(f"Tuning and training {name}...")
    clf = GridSearchCV(config['model'], config['grid'], cv=5, scoring='accuracy', n_jobs=-1)
    clf.fit(X_train_scaled, y_train)
    
    best_model = clf.best_estimator_
    print(f"Best Parameters: {clf.best_params_}")
    
    # Save the model file
    model_filename = f"{name.lower().replace(' ', '_')}.joblib"
    joblib.dump(best_model, model_filename)
    
    # Predict
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    mcc = matthews_corrcoef(y_test, y_pred)
    
    # Store results
    results.append({
        'ML Model Name': name,
        'Accuracy': acc,
        'AUC': auc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'MCC': mcc
    })
    confusion_matrices[name] = confusion_matrix(y_test, y_pred)

### 4. Results & Comparison

In [ ]:
df_results = pd.DataFrame(results)
df_results

#### Visualize Model Performance Comparison

In [ ]:
df_melted = df_results.melt(id_vars='ML Model Name', var_name='Metric', value_name='Score')
plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x='ML Model Name', y='Score', hue='Metric')
plt.title('Comparison of Models on Various Evaluation Metrics')
plt.ylim(0.4, 1.05)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### Plot Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, (name, cm) in enumerate(confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
    axes[i].set_title(f'Confusion Matrix - {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')

# Hide the 6th empty subplot
fig.delaxes(axes[5])
plt.tight_layout()
plt.show()